# Trainable GP and MOGP models

This notebook compares fixed and learned hyperparameters for the single-output GP and the offline replay simulator.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from gp_foundations.gp import GaussianProcessRegressor
from gp_foundations.kernels import MaternKernel
from gp_foundations.multioutput import CoregionalizationMatrix
from gp_foundations.wifi_research import JointStrategySimulator, StrategyObservation

rng = np.random.default_rng(10)
X = np.linspace(0.0, 1.0, 12)[:, None]
true_kernel = MaternKernel(length_scale=0.18, variance=1.3)
y = rng.multivariate_normal(np.zeros(X.shape[0]), true_kernel(X, X) + 0.03 * np.eye(X.shape[0]))

gp = GaussianProcessRegressor(MaternKernel(length_scale=0.75, variance=0.3), noise=0.2).fit(X, y)
initial_objective = gp.negative_log_marginal_likelihood()
result = gp.optimize_hyperparameters(maxiter=60)
learned_objective = gp.negative_log_marginal_likelihood()
assert learned_objective <= initial_objective + 1e-6

X_test = np.linspace(0.0, 1.0, 200)[:, None]
posterior = gp.posterior(X_test)
fig, ax = plt.subplots(figsize=(6.2, 3.4))
ax.plot(X_test[:, 0], posterior.mean, color='#0072B2', label='learned posterior mean')
ax.fill_between(X_test[:, 0], posterior.mean - posterior.std, posterior.mean + posterior.std, color='#0072B2', alpha=0.18)
ax.scatter(X[:, 0], y, color='#D55E00', s=20, label='training observations')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('Single-output GP after likelihood-based training')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.legend(frameon=False)
plt.tight_layout()
print({'initial_objective': initial_objective, 'learned_objective': learned_objective, 'result': result})

grid = np.linspace(0.0, 1.0, 11)
def reward_a(intensity):
    return 1.0 - 2.4 * (intensity - 0.7) ** 2
def reward_b(intensity):
    return 0.95 - 2.2 * (intensity - 0.7) ** 2

observations = [
    StrategyObservation('strategy_a', 0.2, reward_a(0.2)),
    StrategyObservation('strategy_b', 0.2, reward_b(0.2)),
    StrategyObservation('strategy_a', 0.5, reward_a(0.5)),
    StrategyObservation('strategy_a', 0.7, reward_a(0.7)),
    StrategyObservation('strategy_a', 0.9, reward_a(0.9)),
    StrategyObservation('strategy_b', 0.5, reward_b(0.5)),
]

common = {
    'strategy_ids': ('strategy_a', 'strategy_b'),
    'intensity_grid': grid,
    'kernel': MaternKernel(length_scale=1.2, variance=0.25),
    'coregionalization': CoregionalizationMatrix.identity(2),
    'noise': 0.2,
}
fixed = JointStrategySimulator(**common)
learned = JointStrategySimulator(**common, enable_built_in_optimization=True, slow_interval=6, optimization_maxiter=35)
fixed.record_many(observations)
learned.record_many(observations)

fixed_post = fixed.posterior_for_strategy('strategy_b')
learned_post = learned.posterior_for_strategy('strategy_b')
fixed_intensity = float(grid[int(np.argmax(fixed_post.mean))])
learned_intensity = float(grid[int(np.argmax(learned_post.mean))])
assert reward_b(learned_intensity) >= reward_b(fixed_intensity)

fig, ax = plt.subplots(figsize=(6.2, 3.4))
ax.plot(grid, fixed_post.mean, color='#999999', label='fixed posterior mean')
ax.plot(grid, learned_post.mean, color='#009E73', label='learned posterior mean')
ax.axvline(fixed_intensity, color='#999999', linestyle='--', alpha=0.7)
ax.axvline(learned_intensity, color='#009E73', linestyle='--', alpha=0.7)
ax.set_xlabel('Intensity')
ax.set_ylabel('Posterior mean reward')
ax.set_title('Offline replay strategy posterior after built-in slow optimization')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.legend(frameon=False)
plt.tight_layout()
print({'fixed_intensity': fixed_intensity, 'learned_intensity': learned_intensity})
plt.show()
